# 10. RAG

Retrieval直接翻译过来即“检索”，本章Retrieval模块包括与检索步骤相关的所有内容，例如数据的获取、切分、向量化、向量存储、向量检索等模块。

官方文档地址：https://docs.langchain.com/oss/python/langchain/retrieval

> ⚠️ **与官方文档不同：1.x 里已经没有“Retrieval 模块”的说法**
>
> 上面第一句话来自 LangChain **v0.1** 的文档（原文：This section of the documentation covers everything related to the retrieval step）。v0.1 把 LangChain 分成 Model I/O、Retrieval、Agents、Chains、Memory 等几个“模块”。
>
> 1.x 的官方文档里，Retrieval 只是一个讲检索概念的页面，相关组件分散在几个包里。笔记列的 5 项，正好对应这个页面 Building blocks 一节的 5 个组件：

| 笔记的说法 | 官方组件 | 核心类 | 所在的包 |
|---|---|---|---|
| 数据的获取 | Document loaders（文档加载器） | `BaseLoader`，产出 `Document` | 接口在 `langchain-core`；具体的加载器在 `langchain-community` 或各厂商的包 |
| 切分 | Text splitters（文本切分器） | `TextSplitter` | `langchain-text-splitters` |
| 向量化 | Embedding models（嵌入模型） | `Embeddings` | 接口在 `langchain-core`；具体模型在 `langchain-openai` 等厂商的包 |
| 向量存储 | Vector stores（向量数据库） | `VectorStore` | 接口和 `InMemoryVectorStore` 在 `langchain-core`；Chroma 等在各自的集成包（如 `langchain-chroma`） |
| 向量检索 | Retrievers（检索器） | `BaseRetriever` | `langchain-core`；v0.1 文档介绍的 ParentDocumentRetriever、EnsembleRetriever 等高级检索器，1.x 移到了 `langchain-classic` |

> 另外，官方的 Retriever 不限于“向量检索”。官方对它的定义是：给一个查询、返回一组文档的接口。它背后可以是向量库，也可以是搜索 API、SQL 数据库等（见 10.2）。

## 10.1 Retrieval模块的设计意义
1. 知识滞后
LLM训练数据有截止日期，无法及时反映最新的信息或动态变化。比如:难以应对诸如”请推荐当前热门影片”等时间敏感性问题。

2. 知识缺失
大型语言模型(LLM)的训练依赖于网络上海量公开的静态数据，而某些特定领域(如企业内部资料、专有技术文档等)或你的私有数据是缺乏的，导致模型回复时生成不准确甚至虚构的回复。

3. 幻觉
LLM在生成回答时，可能会“胡言乱语”，这种现象称之为LLM的“幻觉”。“幻觉”可以体现为错误陈述、编造事实、错误的复杂推理或者复杂语境下理解能力不足等。

**幻觉问题的严重性**

* 大模型生成内容的不可控，尤其是在金融和医疗领域等领域，一次金额评估的错误，一次医疗诊断的失误，哪怕只出现一次都是致命的。但，对于非专业人士来说可能难以辨识。目前还没有能够百分之百解决这种情况的方案。

**幻觉产生的原因**

* 训练知识存在偏差，这些错误信息被LLM学习后在输出中复现
* LLM训练时过度泛化，将普通的模式应用在特定场合导致不准确输出
* LLM本身没有真正学习到训练数据中深层次的含义，导致在一些需要深入理解或复杂推理的任务中出错
* LLM 缺乏某些领域的相关知识，在面临这些领域的相关问题时编造不存在的信息

**当前大家普遍达成共识的一个方案**:
* 首先，为大模型提供一定的上下文信息，让其输出会变得更稳定。
* 其次，利用本章的RAG，将检索出来的文档和提示词输送给大模型，生成更可靠的答案。

这两点是一回事：第二点是第一点的具体做法。RAG 自动从资料里找出和问题相关的内容，作为上下文交给模型。

> ⚠️ **与官方文档不同：官方列的是两个局限，其中“上下文有限”笔记里没有**
>
> 官方 Retrieval 页开头说，LLM 有两个关键局限：

| 官方列出的局限 | 含义 | 对应笔记 |
|---|---|---|
| **Static knowledge**（知识是静态的） | 训练数据停在某个时间点 | 1. 知识滞后 |
| **Finite context**（上下文有限） | 模型不能一次读入整个资料库 | 笔记没有 |

> * **知识缺失**：Retrieval 页没有单独列出。官方 RAG 教程的开头提到了它：要让模型用上私有数据、最新数据，或训练集之外的数据。
> * **幻觉**：官方没有把它列为局限，只说检索能让回答 grounded（有依据）。RAG 能减少幻觉，但不能消除：模型仍可能误读或忽略检索到的资料（见 10.3）。
>
> **为什么“上下文有限”重要**：就算手里有公司的全部文档，也不能每次提问都把它们整个放进提示词。资料可能远远超过模型的上下文窗口；即使放得下，费用和延迟也会很高。所以要先从资料里**挑出和问题相关的几段**，这一步就是检索。RAG 里的 R 不只是“补充知识”，也是在解决“放不下”的问题。


## 10.2 什么是RAG
RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合 信息检索 （Retrieval）与 文本生成 （Generation）的技术，旨在提升大语言模型在回答专业问题时的 准确性 和 可靠性 。

**典型的检索流程如下**:

![RAG 的一次问答](assets/01-rag-qa-flow.svg)

图 B 中的三步，正好是 RAG 这三个词：

| 字母 | 英文 | 做什么 |
|---|---|---|
| R | Retrieval（检索） | 拿用户的问题去知识库里，找出最相关的几段资料 |
| A | Augmented（增强） | 把找到的资料和问题拼进同一个提示词。被增强的是这次请求的内容，模型本身没有变 |
| G | Generation（生成） | 大模型读这个提示词，依据资料生成回答 |

对比图中的 A、B 两部分：

* 两次请求用的是同一个问题、同一个模型。B 能答对，只是因为这次请求里多了检索到的两段资料。这和第九章 01 笔记的结论一样：模型知道什么，只取决于本次请求发给它的内容。
* B 的回答能注明“依据第 4.2、4.5 条”，因为检索结果带着出处。RAG 就是这样减少幻觉的：让模型照着资料回答，并且让人能回到原文核对。

> ⚠️ **官方文档的两点说明，笔记里没有**
>
> 1. **检索（Retrieval）和 RAG 不是一回事。** 官方说的检索，是在运行时为 LLM 取得相关的上下文，也就是图中的 ①。RAG 是在检索之后接上生成，产出有依据的回答，也就是 ①②③ 合起来。所以官方说检索流程是 RAG 的基础。
> 2. **知识库（knowledge base）不一定是向量数据库。** 官方对知识库的定义是：检索时使用的文档或结构化数据的仓库。如果已经有 SQL 数据库、CRM、内部文档系统，**不需要**把它们重建成向量库，可以：
>    * 把它包装成工具交给 Agent，由 Agent 决定何时查询（Agentic RAG，见 10.5）；
>    * 或者每次先查询它，再把结果作为上下文交给 LLM（2-Step RAG，见 10.5）。
>
>    只有需要从自己的文件搭一个知识库时，才用文档加载器、嵌入模型和向量库来建，这就是 10.4 讲的流程。


## 10.3 RAG的优缺点

> ⚠️ **官方文档没有这一节**：官方 Retrieval 页没有列 RAG 的优缺点，只在 RAG architectures 一节比较了三种架构的可控性、灵活性和延迟（见 10.5）。下面是常见的归纳。每一条都标出了它对应 10.1 的哪个问题、10.4 的哪个环节，方便以后对照。

**优点**

| 优点 | 说明 | 解决 10.1 的哪个问题 |
|---|---|---|
| 知识可以随时更新 | 改知识库里的文档就行，不用重新训练模型 | 知识滞后 |
| 能用上私有、专业的数据 | 公司文档放在自己的知识库里，不进入模型参数；还可以按用户的权限过滤检索结果 | 知识缺失 |
| 回答有依据、可以核对 | 检索结果是带 `metadata`（来源、页码等）的 `Document`，可以要求模型注明出处，人能回到原文核对 | 幻觉 |
| 只发送相关的片段 | 每次只把最相关的 k 个片段放进提示词，而不是整份资料 | 上下文有限 |
| 成本低、组件可替换 | 不需要训练模型的算力和数据。官方特别强调：加载器、切分器、嵌入模型、向量库都可以单独替换，不用改应用逻辑 | — |

**缺点**

| 缺点 | 说明 | 问题出在哪个环节（见 10.4） |
|---|---|---|
| 效果受检索质量限制 | 相关的资料没检索到，或者检索到的不相关，模型就答不好，甚至照着错误的资料回答 | 环节6 检索 |
| 切分会破坏上下文 | 块切得太小，一句话的前提可能被切到别的块；切得太大，相关内容被无关文字稀释。表格、跨页的内容尤其难切 | 环节3 切分 |
| 不擅长需要通读全部资料的问题 | 例如“总结这 100 份合同的共同风险”：每次只取回 k 个片段，看不到全貌 | 环节6 检索 |
| 精确的词匹配不准 | 向量检索按语义相似度找。型号、编号、人名这类要求字面一致的词，可能找不准，常常要配合关键词检索 | 环节4 向量化、环节6 检索 |
| 系统更复杂、请求更慢 | 多了向量化、向量库查询这些步骤；文档更新后，还要重新切分、向量化，保证向量库和原文档一致 | 环节2–5 |
| 不能完全消除幻觉 | 模型可能忽略资料、误读资料，或者把几个片段错误地拼在一起 | 环节7 生成 |


## 10.4 RAG工作流程

RAG 的工作流程分成两个阶段：

* **阶段一 · 索引**（环节1–5）：提前把资料处理好，存进向量库。离线执行，文档有变化时再更新。
* **阶段二 · 检索与生成**（环节6–7）：每次用户提问时在线执行。从向量库取出相关片段，交给大模型生成回答。

10.2 图 B 里的知识库，就是阶段一建出来的；图 B 的 ①②③，就是阶段二。

![RAG 工作流程](assets/02-rag-pipeline.svg)

> ⚠️ **与官方文档不同：环节怎样划分**
>
> 笔记的“环节1：Source”来自 v0.1 文档的流程图。那张图有 6 个环节：**Source → Load → Transform → Embed → Store → Retrieve**，到检索就结束了，因为 v0.1 的 Retrieval 页只讲检索这一步。1.x 官方 Retrieval 页的流程图有三处不同：
>
> 1. **多了生成**：Retriever 后面接着 “LLM uses retrieved info → Answer”，检索和生成画在同一张图里。上图把它记作环节7。
> 2. **画出了提问这一侧**：用户问题先做 Query embedding（问题向量化），再到向量库里搜索。上图把它放在阶段二。
> 3. **Transform 改成了 Split into chunks**：v0.1 的 Transform 泛指对文档做转换，切分是其中最主要的一种；1.x 只写切分，对应的组件是 Text splitters。
>
> “索引”和“检索与生成”这两个阶段名，来自官方之前的 RAG 教程（Indexing 和 Retrieval and generation）。1.x 的图里没有写阶段名，但同样分成两部分：一部分导入资料，一部分处理提问。

### 阶段一：索引（Indexing）

#### 环节1：Source（数据源）

知识从哪里来。可以是本地文件（PDF、Word、Markdown、CSV），网页，Notion、Slack、Google Drive 这类在线服务，也可以是数据库。来源不同，格式差别很大，所以下一步要先统一格式。

#### 环节2：Load（加载）

**文档加载器（Document Loaders）** 从数据源读出内容，统一转换成 LangChain 的 `Document` 对象。`Document` 有三个属性：

| 属性 | 类型 | 含义 |
|---|---|---|
| `page_content` | `str` | 文本内容 |
| `metadata` | `dict` | 任意的元数据，例如来源文件、页码。检索结果能注明出处，靠的就是它 |
| `id` | `str`（可选） | 文档的标识 |

例如加载一个 PDF，常见的做法是每页生成一个 `Document`，`metadata` 里记下文件名和页码。

#### 环节3：Split（切分）

**文本切分器（Text Splitters）** 把长文档切成小块（chunk）。切出来的仍然是 `Document`，原文档的 `metadata` 会复制到每一块上。

为什么要切：

* 一整页、一整篇对检索来说太粗。真正相关的可能只有一两句话，被周围的无关文字“稀释”后，整页的向量就不能准确代表这一两句话；
* 每一块要能单独被检索出来，并且放得进模型的上下文窗口。

两个最常用的参数：

* `chunk_size`：每块的最大长度（默认按字符数计算）；
* `chunk_overlap`：相邻两块重叠的长度。一句话正好被切断时，两边的块都还保留着一部分上下文。

官方推荐通用文本使用 `RecursiveCharacterTextSplitter`：它依次尝试按段落（`"\n\n"`）、换行（`"\n"`）、空格切，最后才按单个字符切，直到每块都不超过 `chunk_size`。

#### 环节4：Embed（向量化）

**嵌入模型（Embedding Models）** 把一段文本变成一个固定长度的数字列表，也就是向量。意思相近的文本，向量也离得近，所以可以用余弦相似度等指标衡量两段文本的意思有多接近。第九章 07 笔记的语义搜索，用的就是这套接口（`init_embeddings`）。

`Embeddings` 接口有两个方法，分别用在两个阶段：

| 方法 | 输入 → 输出 | 什么时候用 |
|---|---|---|
| `embed_documents(texts)` | `list[str]` → `list[list[float]]` | 阶段一：把所有 chunk 向量化 |
| `embed_query(text)` | `str` → `list[float]` | 阶段二：把用户问题向量化 |

#### 环节5：Store（存储）

**向量数据库（Vector Stores）** 把每一块的向量、原文和 `metadata` 存在一起，并且能按向量做相似度搜索。学习时可以用内存里的 `InMemoryVectorStore`；实际项目里用本地运行的 Chroma、Milvus，或者 PGVector、Pinecone 这类服务。

写入时调用 `vector_store.add_documents(chunks)`。向量库在创建时就拿到了嵌入模型，写入时它自己调用 `embed_documents` 完成向量化，所以环节4和环节5在代码里通常是同一行。


### 阶段二：检索与生成（Retrieval and generation）

#### 环节6：Retrieve（检索）

用户提问时：

1. 用**同一个**嵌入模型把问题向量化（`embed_query`）。一定要是同一个模型：不同模型的向量空间不同，长度也可能不同，放在一起比较没有意义。
2. 在向量库里找出和问题向量最相似的 k 块（top-k），连同原文和 `metadata` 一起返回。

LangChain 用**检索器（Retriever）** 统一这一步：输入一个字符串，返回 `list[Document]`。向量库调用 `as_retriever()` 就能得到一个检索器，`search_type` 有三种取值：

| `search_type` | 含义 |
|---|---|
| `"similarity"`（默认） | 只按相似度取前 k 块 |
| `"mmr"` | 最大边际相关性：兼顾相关性和多样性，避免取回几块几乎一样的内容 |
| `"similarity_score_threshold"` | 只返回相似度超过阈值的块 |

检索器和向量库的区别：

* 检索器是 Runnable（见第六章 01 笔记 2.1.3），有 `invoke`、`batch` 等标准方法，可以用 `|` 和其他组件串起来；向量库不是 Runnable。
* 检索器背后不一定是向量库，也可以包装搜索 API 等其他来源。

#### 环节7：Generate（生成）

把检索到的片段和用户问题拼进提示词（例如“请只根据下面的资料回答……”），交给聊天模型生成回答，也就是 10.2 图 B 的 ② 和 ③。v0.1 的 6 环节图没有这一步，1.x 的流程图把它画了进来。


### 各环节小结与学习重点

| 环节 | 做什么 | LangChain 组件 | 输入 → 输出 | 学习重点 |
|---|---|---|---|---|
| 1 Source | 确定资料来源 | — | — | 了解 |
| 2 Load | 读取资料，统一成 `Document` | Document Loaders | 文件 → `list[Document]` | ⭐ `Document` 的结构；常用的加载器 |
| 3 Split | 切成小块 | Text Splitters | 长 `Document` → 多个小 `Document` | ⭐⭐ `chunk_size`、`chunk_overlap` 怎样影响检索效果 |
| 4 Embed | 文本 → 向量 | Embeddings | `str` → `list[float]` | ⭐ 向量和语义相似度；两个方法分别用在哪个阶段 |
| 5 Store | 保存向量，支持相似度搜索 | Vector Stores | chunk → 向量库 | ⭐⭐ 写入、相似度搜索、按 `metadata` 过滤 |
| 6 Retrieve | 找出最相关的 k 块 | Retrievers | 问题 → `list[Document]` | ⭐⭐ `as_retriever()` 和 `search_type` |
| 7 Generate | 依据资料回答 | Chat Model + 提示词 | 问题 + 片段 → `AIMessage` | ⭐ 提示词怎么拼；和 Agent 结合见 10.5 |

⭐⭐ 是本章的重点，直接决定 RAG 答得好不好；⭐ 需要会用；“了解”知道是什么就行。


## 10.5 RAG的三种架构

> ⚠️ **笔记原本没有这一节**：官方 Retrieval 页的后半部分讲 RAG 的三种架构（RAG architectures），这里补上。10.4 的工作流程属于其中最基础的 2-Step RAG。

三种架构的区别在于：**检索什么时候发生、由谁决定。**

![三种 RAG 架构](assets/03-rag-architectures.svg)

官方的对比表：

| 架构 | 说明 | 可控性 | 灵活性 | 延迟 | 官方举的场景 |
|---|---|---|---|---|---|
| **2-Step RAG** | 检索总是发生在生成之前，简单、可预期 | ✅ 高 | ❌ 低 | ⚡ 快 | FAQ、文档问答机器人 |
| **Agentic RAG** | 由 LLM 驱动的 Agent 在推理过程中决定**何时**检索、**怎样**检索 | ❌ 低 | ✅ 高 | ⏳ 不固定 | 能调用多个工具的研究助手 |
| **Hybrid RAG** | 结合前两者的特点，并加入验证步骤 | ⚖️ 中 | ⚖️ 中 | ⏳ 不固定 | 需要质量校验的专业领域问答 |

### 10.5.1 2-Step RAG

每次都先检索、再生成。检索写死在程序里，模型不参与决定要不要检索。适合“回答之前一定要先查资料”的场景。

官方对延迟的说明：2-Step RAG 的延迟更**可预期**，因为 LLM 的调用次数是已知的、有上限的。这个结论的前提是 LLM 推理时间占大头；实际延迟还受检索环节影响，例如 API 响应时间、网络延迟、数据库查询。

### 10.5.2 Agentic RAG

官方的提示：Agent 要具备 RAG 能力，唯一需要的是一个或多个**能获取外部知识的工具**，例如文档加载器、Web API、数据库查询。

所以第七章学的 `create_agent` + 工具，就是 Agentic RAG 的写法：把检索写成工具，由模型决定要不要调用、调用几次、传什么参数。官方的例子是一个抓取网页的工具（下面是官方代码的主要部分，没有运行）：

```python
import requests
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text

agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[fetch_url],  # 用于检索的工具
    system_prompt=system_prompt,
)
```

这个例子里没有向量库，“检索”就是抓取网页，这正是 10.2 说的“知识库不一定是向量库”。如果要用 10.4 建好的向量库，就把检索器包装成一个工具交给 Agent。

### 10.5.3 Hybrid RAG

结合前两种架构，在流程中加入中间步骤。官方列出的典型组成：

| 步骤 | 做什么 | 不满意时 |
|---|---|---|
| 查询增强（Query enhancement） | 改写不清楚的问题、生成几种不同的问法，或者给问题补充上下文，提高检索质量 | — |
| 检索验证（Retrieval validation） | 检查检索到的文档是否相关、是否足够 | 改写查询，重新检索 |
| 答案验证（Answer validation） | 检查答案是否准确、完整、和资料一致 | 重新生成或修改答案 |

这几步之间可以多次循环（图中右侧的两条回路）。官方说它适合三类场景：问题模糊或描述不完整；需要验证、质量控制；要用多个数据源或反复改进。官方配套的示例教程（Agentic RAG with Self-Correction）是用 LangGraph 写的。

> 注意区分 **Hybrid RAG** 和**混合检索**（hybrid search）：前者说的是 RAG 的流程怎样组织；后者说的是环节6用什么方法检索，指把关键词检索和向量检索结合起来（对应 10.3 缺点里的“精确的词匹配不准”）。中文资料里两者都常叫“混合”，看资料时要分清是哪一个。

### 10.5.4 学习重点

| 架构 | 学习重点 |
|---|---|
| 2-Step RAG | ⭐⭐ 本章主线：先把 10.4 的每个环节学会 |
| Agentic RAG | ⭐ 把检索器包装成工具交给 Agent，用到第五章的工具和第七章的 Agent |
| Hybrid RAG | 了解：知道有哪几种验证步骤即可，官方示例用 LangGraph 实现 |
